# Introduction to OpenMP

## Serial to Parallel: OpenMP

The simplest and most common parallel pattern is to take a serial program and convert it into a parallel program.

* My code's not running fast enough:
  * Interactive AI: a slow model makes a chat or agent response feel laggy
  * Video/streaming: encode/decode falls behind real time, frames drop
  * Data pipelines: ETL, indexing, or training runs don't finish in the batch window
  * High-frequency trading: a slower model loses the trade to a faster one -> lost arbitrage opportunity
* This leads to a natural software engineering process:
  * Profile code: find out what's slow
  * Parallelize the slow part(s) only
  * Migrate from a serial implementation to a parallel implementation
* It's not the best process:
  * Serial-to-parallel doesn't produce the best designs
  * The best parallel implementation may require a totally different design, not an incremental refactor of the serial one
* Just the easiest.

### What is OpenMP?

A parallel programming environment (not a language) for:
* Fork/join execution model
* Loop parallelism patterns
* Thread parallelism on shared-memory architectures

It's the simplest approach to parallelism:
* Write a serial program in a language you already know (C/C++ or Fortran)
* Add directives (`#pragma`) to parallelize portions of the code
* Get a parallel program that computes the same result -- *serial-to-parallel equivalence*

Merits:
* Incremental parallelism
* Simple to use
* Portable (for the most part)

Limitations:
* Difficult to manage memory usage
* No distributed-memory capability

### The OpenMP Toolchain

No separate toolchain -- OpenMP directives compile with your regular C compiler:

* Add `-fopenmp` to the compiler command line
  * generates code from `#pragma omp ...` directives
  * links against `libomp`
* `#include <omp.h>` to get runtime functions (`omp_get_thread_num()`, `omp_set_num_threads()`, ...)

This directory's examples compile with clang + Homebrew's libomp on macOS (see [Makefile](Makefile)):

```
clang -Xpreprocessor -fopenmp -O3 \
  -I/opt/homebrew/opt/libomp/include \
  -L/opt/homebrew/opt/libomp/lib -lomp \
  hello_omp.c -o hello_omp
```

On Linux with gcc it's simpler: `gcc -fopenmp -O3 program.c -o program`.

#### (Aside) Compiler optimization

All compilers, including gcc/clang, need their optimization flags set explicitly to get good performance.

* `-O0` (default) -- no optimization; fastest to compile, best for debugging
* `-O1` -- simple optimizations that don't cost much compile time
* `-O2` -- rewrite loops, follow jump pointers, inline small functions
* `-O3` -- vectorize, inline aggressively, branch prediction

Debug at `-O0` so the code makes sense in a debugger; benchmark at `-O3`. All examples in this directory build at `-O3`.

### Memory Model and Hardware

OpenMP creates *threads* that run on multiple processor cores, all sharing one address space.

* **Shared memory** -- coherent read/write to common memory from multiple cores
  * Coherent = repeatable reads, a read sees the last write, ...
  * Abstraction: there is a single memory for all processors
  * Threads communicate by reading and writing ordinary memory -- no message passing

This is why the OpenMP primitives look the way they do: instead of sending data between processes, a directive tells the compiler *which threads touch which memory*, and the runtime handles the rest.

## The Primitives

Each primitive below has a minimal, standalone example in this directory. Build any of them with `make <name>` from `openmp/`, or `make` to build everything.

| primitive | pragma | file |
|---|---|---|
| parallel region | `#pragma omp parallel` | [hello_omp.c](hello_omp.c) |
| parallel loop | `#pragma omp parallel for` | [vector_add.c](vector_add.c) |
| reduction | `reduction(+:sum)` clause | [reduction.c](reduction.c), [dot_product.c](dot_product.c) |
| task | `#pragma omp task` / `taskwait` | [fib.c](fib.c) |

These mirror the primitives in [../cilk/](../cilk/) (`cilk_spawn`/`cilk_sync`, `cilk_for`, `cilk_reducer`), so you can compare the two models side by side. Scheduling policy (`schedule(static/dynamic/guided)`) and explicit synchronization (`critical`/`atomic`/locks) are separate topics for later notebooks -- every example here just uses OpenMP's defaults.

### `#pragma omp parallel` -- Creating a Team

The fundamental OpenMP primitive: fork a *team* of threads that all execute the following block.

```c
#pragma omp parallel
{
    int tid = omp_get_thread_num();
    int nthreads = omp_get_num_threads();
    printf("Hello from thread %d of %d\n", tid, nthreads);
}
```

The block `{ ... }` is what gets replicated -- every thread in the team runs a copy of it. Every other OpenMP construct (`parallel for`, `task`, ...) has to run *inside* a region like this one; `parallel` is what creates the thread team that later directives hand work to.

Full example: [hello_omp.c](hello_omp.c)

In [ ]:
!make hello_omp && ./hello_omp

### `#pragma omp parallel for` -- Loop Parallelism

`parallel for` is a work-sharing construct: it forks a team (like `parallel` above) *and* splits the loop's iterations across it. Iterations must be independent -- no iteration may read a value another writes.

```c
#pragma omp parallel for
for (long i = 0; i < N; i++)
    c[i] = a[i] + b[i];       // each i touches a different slot
```

By default the runtime divides the index range into one contiguous chunk per thread, handed out up front. (Later notebooks look at other scheduling policies for when the work per iteration isn't uniform.)

Full example: [vector_add.c](vector_add.c)

In [ ]:
!make vector_add && ./vector_add 10000000

### The `reduction` Clause -- Safe Accumulation

`sum += a[i]` inside a `parallel for` is a race: every thread updates the same location. The `reduction` clause gives each thread a private copy of `sum`, initialized to the operator's identity element, and combines the copies after the loop.

```c
double sum = 0.0;
#pragma omp parallel for reduction(+:sum)
for (long i = 0; i < N; i++)
    sum += a[i];
```

OpenMP predefines the identity/combine behavior for a fixed set of operators (`+, -, *, &, |, ^, &&, ||, min, max`); anything outside that set needs the `critical`/`atomic`/lock primitives covered later.

Full examples: [reduction.c](reduction.c) (the pattern in isolation) and [dot_product.c](dot_product.c) (same clause, benchmarked serial-vs-parallel in GB/s).

In [ ]:
!make reduction dot_product && ./reduction 10000000 && ./dot_product 10000000

### `#pragma omp task` / `taskwait` -- Dynamic Parallelism

`parallel` and `parallel for` both need to know the work up front. `task` creates work dynamically -- useful for recursive divide-and-conquer, where the amount of parallelism isn't known until runtime.

```c
long a, b;
#pragma omp task shared(a)
a = fib(n - 1);       // may run in parallel ...
b = fib(n - 2);       // ... with this continuation

#pragma omp taskwait   // both must finish before we add
return a + b;
```

Two things about `task` that don't come up with `parallel`/`parallel for`:

* Tasks must be generated from inside a `parallel` region, and normally from a single thread (`#pragma omp single`) -- otherwise every thread in the team redundantly runs the whole recursion.
* A variable written inside a task and read after `taskwait` must be marked `shared`. The default for task-local variables is `firstprivate` (a private copy), so without the clause the write is silently lost.

Full example: [fib.c](fib.c)

In [ ]:
!make fib && ./fib 30

## Summary

| you want to... | use | file |
|---|---|---|
| run a block on every thread | `#pragma omp parallel` | [hello_omp.c](hello_omp.c) |
| split independent loop iterations across threads | `#pragma omp parallel for` | [vector_add.c](vector_add.c) |
| accumulate a value across threads without a race | `reduction(op:var)` | [reduction.c](reduction.c), [dot_product.c](dot_product.c) |
| create work dynamically (recursion) | `#pragma omp task` / `taskwait` | [fib.c](fib.c) |

Next up: scheduling policies (`schedule(static/dynamic/guided)`) and explicit synchronization (`critical`/`atomic`/locks) -- both build directly on the primitives above.